# 05 Brazil GWR

Select **Runtime → Change runtime type → NVIDIA GPU**, then run all cells
in a fresh session. Each notebook installs its own immutable public package
checkout and verifies every input. No Python-source upload is required.

Package source: `Favioleiva/gpugwr@7298c5629a48e7b875739f117a27d2e00f2512de`. These links become usable only
after owner-authorized publication. Local tests are not Colab verification.
A failed GPU request or numerical comparison stops execution visibly.
Save the executed notebook and download the output ZIP as evidence.


In [ ]:
import json
import os
import subprocess
import sys
import tempfile
from pathlib import Path

SOURCE_COMMIT = "7298c5629a48e7b875739f117a27d2e00f2512de"
REPOSITORY = "https://github.com/Favioleiva/gpugwr.git"
RUN = Path(tempfile.mkdtemp(prefix="gpugwr-"))
CHECKOUT = RUN / "source"
OUTPUT = RUN / "outputs"
OUTPUT.mkdir()
subprocess.run(["git", "init", str(CHECKOUT)], check=True)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "remote",
        "add",
        "origin",
        REPOSITORY,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "fetch",
        "--depth",
        "1",
        "origin",
        SOURCE_COMMIT,
    ],
    check=True,
)
subprocess.run(
    [
        "git",
        "-C",
        str(CHECKOUT),
        "checkout",
        "--detach",
        "FETCH_HEAD",
    ],
    check=True,
)
os.environ["CUPY_CACHE_DIR"] = str(RUN / "cupy-cache")
with (OUTPUT / "bootstrap.log").open("w", encoding="utf-8") as log:
    completed = subprocess.run(
        [
            sys.executable,
            str(CHECKOUT / "Colab/bootstrap.py"),
            "--checkout",
            str(CHECKOUT),
            "--commit",
            SOURCE_COMMIT,
            "--output",
            str(OUTPUT / "bootstrap.json"),
        ],
        stdout=log,
        stderr=subprocess.STDOUT,
        text=True,
    )
print((OUTPUT / "bootstrap.log").read_text("utf-8")[-12000:])
completed.check_returncode()


In [ ]:
import numpy as np
import pandas as pd
from IPython.display import display
from threadpoolctl import threadpool_limits
from gpuGWR import GWRSpatialCache, gpuGWR
from gpugwr.notebook_measurement import environment, measure
from gpugwr.reproducibility import (
    ReproductionAssets,
    export_geography,
    model_options,
    validate_arrays,
    validate_fit,
    write_json,
    zip_outputs,
)

assets = ReproductionAssets(CHECKOUT, RUN / "data-cache")
runtime = environment()
assert runtime["gpu"]["available"], (
    "A genuine GPU is required; no CPU fallback."
)
runtime["source_commit"] = SOURCE_COMMIT
write_json(OUTPUT / "environment.json", runtime)
print(json.dumps(runtime, indent=2))


## Brazil: literal historical reproduction, 5,562 municipalities

Response `DfHRElc` and the four original predictor sets are displayed below.
Existing logged columns are not transformed again. All models use an
intercept, adaptive bisquare weights and **1,345 neighbors**, without
standardization or bandwidth selection. `COD_MUN_I` identifies municipalities.
Variable names are preserved; the source does not establish a complete
authoritative units dictionary, so units are not invented.

**Historical coordinate convention:** polygon centroids computed directly
in EPSG:4326, then Euclidean distances in degrees. This intentionally
reproduces the original experiment; it is not a geographically corrected
distance model. No corrected alternative is mixed into these results.

The 34.8 MB GeoJSON is requested from the original pinned Gist and must
match the approved SHA-256. Its upstream availability/identity was not
established in the local historical record. Acquisition failure is a
failed reproducibility step, never permission to substitute other data.


In [ ]:
frame, specification = assets.load_case("brazil")
print(json.dumps(specification, indent=2))
assert len(frame) == 5562
assert specification["bandwidth"] == 1345
display(frame.head())


def fit_sequence(reuse):
    """Time cache preparation, all four fits and cache release."""
    cache = None
    fitted_models = {}
    if reuse:
        cache = GWRSpatialCache(
            frame[specification["coordinates"]].to_numpy(),
            bandwidth=1345,
            kernel="bisquare",
            fixed=False,
            engine="gpu",
        )
    try:
        for model in specification["models"]:
            options = model_options(specification, model=model)
            fitted_models[model["name"]] = gpuGWR(
                frame,
                **options,
                engine="gpu",
            ).fit(spatial_cache=cache)
    finally:
        if cache is not None:
            cache.close()
    return fitted_models


timings = []
independent = None
with threadpool_limits(limits=1):
    for reuse in (False, True):
        fitted_models, timing = measure(
            lambda: fit_sequence(reuse), "gpu"
        )
        label = "reused" if reuse else "independent"
        timing.update(
            {"reuse": reuse, "dataset": "brazil", "models": {}}
        )
        for model in specification["models"]:
            name = model["name"]
            result = fitted_models[name]
            folder = OUTPUT / label / name
            result.export(folder)
            export_geography(
                result, frame, specification, assets, folder
            )
            validate_fit(
                result,
                assets.acquire(model["reference"]),
                folder / "canonical_validation.json",
            )
            assert (
                result.bandwidth == 1345
                and result.n_observations == 5562
            )
            if reuse:
                validate_arrays(
                    result.numerical,
                    independent[name].numerical,
                    folder / "reuse_validation.json",
                )
            timing["models"][name] = result.configuration
            print(label, name, result.summary())
            display(result.local_results().head())
        if not reuse:
            independent = fitted_models
        timings.append(timing)
        write_json(OUTPUT / "performance.json", timings)
display(
    pd.DataFrame(
        [
            {
                "reuse": item["reuse"],
                "four_model_seconds": item["wall_seconds"],
                "sampled_peak_mib": item[
                    "sampled_peak_device_memory_mib"
                ],
            }
            for item in timings
        ]
    )
)


In [ ]:
archive = zip_outputs(OUTPUT)
print(f"Evidence archive: {archive}")
try:
    from google.colab import files
except ImportError:
    from IPython.display import FileLink

    display(FileLink(str(archive)))
else:
    files.download(str(archive))
